# NeuroBridge-S4 Graph Learning — Phase 8: Reference-Calibrated Trajectory Envelope

> **The within-subject self-baseline remains primary.** Phase 8 adds a *secondary calibration layer*: an expected-variability envelope that says whether a subject's own baseline-relative graph change is small, moderate, or unusually large.

```text
Primary comparison:    Current subject state vs that subject's own baseline.
Secondary calibration: Reference/analog cohort variability for expected
                       fluctuation envelope, noise scale, and rarity context.
Interpretation layer:  NASA HRP five hazards as operational context,
                       not exposure measurement.
```


## 1. Plain-language purpose

Phase 8 keeps self-baseline tracking as the primary method but adds a reference-calibrated variability envelope. This helps determine whether a subject's own graph change is small, moderate, or unusually large relative to expected variability.

The HRP-facing question is: *Is this within-subject graph change larger than expected biological/measurement variability, and which domains or graph features drive it?*

## 2. Why this is not a return to population-normal comparison

This phase does not ask whether a subject is normal compared with healthy people. It asks whether the subject's own change from baseline is larger than expected under a reference-calibrated variability envelope.

> The reference envelope does not define whether a person is healthy or unhealthy. It calibrates how large a within-subject graph change is relative to expected variability in available proxy or analog data.

## 3. Scientific guardrails

- This is **not** actual Artemis II data unless such data are explicitly provided.
- If example envelope data are used, they are **schema demonstration only**.
- Envelope exceedance is **not** diagnosis.
- Envelope exceedance is **not** health risk scoring.
- Hazard-context envelope does **not** measure exposure.
- Reference data calibrate **expected variability, not health status**.

## 4. Setup

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import pandas as pd

from neurobridge_graph.reference_envelope import (
    load_reference_envelope_inputs,
    create_example_reference_envelope,
    build_envelope_from_reference_deltas,
    build_envelope_from_summary_table,
    score_node_deltas_against_envelope,
    score_graph_deltas_against_envelope,
    score_hazard_deltas_against_envelope,
    build_phase8_envelope_summary,
    CORE_ENVELOPE_STATEMENT,
)
from neurobridge_graph.envelope_visualization import (
    plot_domain_delta_envelope,
    plot_graph_metric_envelope,
    plot_hazard_delta_envelope,
    plot_envelope_exceedance_heatmap,
    plot_reference_envelope_overview,
)
from neurobridge_graph.envelope_reporting import generate_phase8_report

RESULTS = repo_root / 'results'
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'
REPORTS = RESULTS / 'reports'
DATA = repo_root / 'data'
for d in [TABLES, FIGURES, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 170)
print('Setup complete.')
print(CORE_ENVELOPE_STATEMENT)


Setup complete.
The reference envelope does not define whether a person is healthy or unhealthy. It calibrates how large a within-subject graph change is relative to expected variability in available proxy or analog data.


## 5. Load Phase 6 and Phase 7 outputs

Core required inputs are the Phase 6 node and graph delta tables. Hazard deltas and the Phase 7 attribution summary are recommended but optional.

In [2]:
CORE = {
    'node_deltas': 'longitudinal_node_deltas.csv',
    'graph_deltas': 'longitudinal_graph_deltas.csv',
}
OPTIONAL = {
    'hazard_deltas': 'longitudinal_hazard_deltas.csv',
    'attribution_summary': 'phase7_attribution_summary.csv',
}

core_missing = [f for f in CORE.values() if not (TABLES / f).exists()]
if core_missing:
    raise SystemExit(
        'Phase 8 requires Phase 6 longitudinal delta outputs. '
        'Please run the Phase 6 notebook first. Missing: ' + str(core_missing)
    )

def _load(fname):
    p = TABLES / fname
    return pd.read_csv(p) if p.exists() else None

node_deltas = _load(CORE['node_deltas'])
graph_deltas = _load(CORE['graph_deltas'])
hazard_deltas = _load(OPTIONAL['hazard_deltas'])
attribution_summary = _load(OPTIONAL['attribution_summary'])

readiness_rows = []
for key, fname in {**CORE, **OPTIONAL}.items():
    p = TABLES / fname
    req = 'required' if fname in CORE.values() else 'optional'
    if p.exists():
        df = _load(fname)
        readiness_rows.append({'file': fname, 'status': 'found', 'rows': len(df),
                               'columns': df.shape[1], 'required_or_optional': req,
                               'notes': 'loaded'})
    else:
        readiness_rows.append({'file': fname, 'status': 'missing', 'rows': 0,
                               'columns': 0, 'required_or_optional': req,
                               'notes': 'blocks Phase 8' if req == 'required'
                                        else 'layer unavailable'})
readiness = pd.DataFrame(readiness_rows)
readiness.to_csv(TABLES / 'phase8_input_readiness_check.csv', index=False)
readiness


,file,status,rows,columns,required_or_optional,notes
0,longitudinal_node_deltas.csv,found,60,12,required,loaded
1,longitudinal_graph_deltas.csv,found,70,10,required,loaded
2,longitudinal_hazard_deltas.csv,found,50,9,optional,loaded
3,phase7_attribution_summary.csv,found,10,10,optional,loaded


## 6. Load or create reference envelope

We look for real reference/analog calibration files under `data/processed/`. If none are present, we create a clearly marked **schema-demonstration** example envelope.

> This example reference envelope is used only to demonstrate the calibration workflow. It is not scientific evidence and should be replaced with real analog/reference variability data.

In [3]:
reference_inputs = load_reference_envelope_inputs(repo_root)
print('Reference files found:', sorted(reference_inputs.keys()) or 'none')

if 'reference_longitudinal_deltas' in reference_inputs:
    envelope = build_envelope_from_reference_deltas(
        reference_inputs['reference_longitudinal_deltas'])
    used_example = False
    provenance_note = 'Calibration source: provided reference/analog longitudinal deltas.'
elif any(k in reference_inputs for k in (
        'reference_domain_variability', 'reference_graph_feature_variability',
        'reference_hazard_variability')):
    parts = [reference_inputs[k] for k in (
        'reference_domain_variability', 'reference_graph_feature_variability',
        'reference_hazard_variability') if k in reference_inputs]
    envelope = build_envelope_from_summary_table(pd.concat(parts, ignore_index=True))
    used_example = False
    provenance_note = 'Calibration source: provided reference/analog variability summaries.'
else:
    example_path = DATA / 'examples' / 'example_reference_variability_envelope.csv'
    example_df = create_example_reference_envelope(example_path)
    envelope = build_envelope_from_summary_table(example_df)
    used_example = True
    provenance_note = (
        'Calibration source: EXAMPLE schema-demonstration envelope only. '
        'Not scientific evidence; replace with real analog/reference variability data.')
    print('No real reference data found. Created example envelope at:')
    print('  ', example_path.relative_to(repo_root))

envelope.to_csv(TABLES / 'reference_trajectory_envelope.csv', index=False)
print(provenance_note)
print(f'Envelope features calibrated: {len(envelope)}')
envelope.head(20)


Reference files found: none
No real reference data found. Created example envelope at:
   data/examples/example_reference_variability_envelope.csv
Calibration source: EXAMPLE schema-demonstration envelope only. Not scientific evidence; replace with real analog/reference variability data.
Envelope features calibrated: 18


,feature,feature_type,group,n_reference,median_delta,mad_delta,lower_q,upper_q,lower_bound,upper_bound,envelope_method,data_type
0,Body composition / physical status,node,NaN,30,0.0,0.050,0.05,0.95,-0.12,0.12,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
1,Cardiovascular regulation,node,NaN,30,0.0,0.050,0.05,0.95,-0.12,0.12,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
2,Hematologic / oxygen-carrying,node,NaN,30,0.0,0.045,0.05,0.95,-0.11,0.11,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
3,Inflammation / immune-adjacent,node,NaN,30,0.0,0.060,0.05,0.95,-0.14,0.14,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
4,Metabolic regulation,node,NaN,30,0.0,0.050,0.05,0.95,-0.12,0.12,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
5,Recovery-related markers,node,NaN,30,0.0,0.050,0.05,0.95,-0.12,0.12,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
6,mean_node_activation,graph,NaN,30,0.0,0.030,0.05,0.95,-0.08,0.08,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
7,max_node_activation,graph,NaN,30,0.0,0.040,0.05,0.95,-0.10,0.10,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
8,total_node_activation,graph,NaN,30,0.0,0.200,0.05,0.95,-0.50,0.50,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence
9,n_active_domains,graph,NaN,30,0.0,0.500,0.05,0.95,-1.00,1.00,example_summary_quantile_mad,schema_demonstration_not_scientific_evidence


## 7. How the envelope is constructed

For each feature (biological domain, graph metric, or hazard context), the envelope summarizes the expected variability of the baseline-relative delta:

- **median delta** — the typical reference delta (center of the envelope);
- **MAD** — median absolute deviation, a robust spread estimate;
- **5th / 95th percentile bounds** — the expected-variability envelope edges;
- **robust z-score** — `(delta − median) / (1.4826 × MAD)`, comparable to a standard z-score;
- **expected variability envelope** — the band a within-subject delta is compared against.

This *supports* self-baseline tracking: it never replaces the personal baseline, it only calibrates how large the within-subject change is.

## 8. Score node deltas against the envelope

Each domain delta is classified as `within_expected_envelope`, `near_envelope_boundary`, `outside_expected_envelope`, or `insufficient_reference`.

In [4]:
node_scores = score_node_deltas_against_envelope(node_deltas, envelope)
node_scores.to_csv(TABLES / 'reference_calibrated_node_delta_scores.csv', index=False)
print('Position counts:', node_scores['envelope_position'].value_counts().to_dict())
outside_nodes = node_scores[node_scores['envelope_position'] == 'outside_expected_envelope']
outside_nodes.sort_values('envelope_exceedance', ascending=False)[
    ['subject_id', 'timepoint', 'domain', 'delta_activation',
     'lower_bound', 'upper_bound', 'robust_z']].head(12)


Position counts: {'within_expected_envelope': 32, 'outside_expected_envelope': 24, 'near_envelope_boundary': 4}


,subject_id,timepoint,domain,delta_activation,lower_bound,upper_bound,robust_z
18,Demo_Crew_01,T3_post,Body composition / physical status,0.55,-0.12,0.12,7.4194
21,Demo_Crew_01,T3_post,Inflammation / immune-adjacent,0.55,-0.14,0.14,6.1828
52,Demo_Crew_02,T3_post,Metabolic regulation,0.45,-0.12,0.12,6.0704
51,Demo_Crew_02,T3_post,Inflammation / immune-adjacent,0.43,-0.14,0.14,4.8339
19,Demo_Crew_01,T3_post,Cardiovascular regulation,0.40,-0.12,0.12,5.3959
12,Demo_Crew_01,T2_inflight,Body composition / physical status,0.40,-0.12,0.12,5.3959
15,Demo_Crew_01,T2_inflight,Inflammation / immune-adjacent,0.40,-0.14,0.14,4.4966
46,Demo_Crew_02,T2_inflight,Metabolic regulation,0.35,-0.12,0.12,4.7214
49,Demo_Crew_02,T3_post,Cardiovascular regulation,0.35,-0.12,0.12,4.7214
50,Demo_Crew_02,T3_post,Hematologic / oxygen-carrying,0.32,-0.11,0.11,4.7964


## 9. Score graph metric deltas against the envelope

Graph metrics with the largest robust z-scores indicate which graph-level features drive the within-subject change beyond expected variability.

In [5]:
graph_scores = score_graph_deltas_against_envelope(graph_deltas, envelope)
graph_scores.to_csv(TABLES / 'reference_calibrated_graph_delta_scores.csv', index=False)
print('Position counts:', graph_scores['envelope_position'].value_counts().to_dict())
graph_scores.reindex(graph_scores['robust_z'].abs().sort_values(ascending=False).index)[
    ['subject_id', 'timepoint', 'metric', 'delta_value',
     'lower_bound', 'upper_bound', 'robust_z', 'envelope_position']].head(12)


Position counts: {'within_expected_envelope': 38, 'outside_expected_envelope': 29, 'near_envelope_boundary': 3}


,subject_id,timepoint,metric,delta_value,lower_bound,upper_bound,robust_z,envelope_position
26,Demo_Crew_01,T3_post,graph_density,0.73333,-0.12,0.12,9.8925,outside_expected_envelope
21,Demo_Crew_01,T3_post,mean_node_activation,0.38834,-0.08,0.08,8.7311,outside_expected_envelope
25,Demo_Crew_01,T3_post,active_domain_fraction,1.00000,-0.20,0.20,8.4311,outside_expected_envelope
24,Demo_Crew_01,T3_post,n_active_domains,6.00000,-1.00,1.00,8.0939,outside_expected_envelope
23,Demo_Crew_01,T3_post,total_node_activation,2.33000,-0.50,0.50,7.8578,outside_expected_envelope
22,Demo_Crew_01,T3_post,max_node_activation,0.45000,-0.10,0.10,7.5880,outside_expected_envelope
56,Demo_Crew_02,T3_post,mean_node_activation,0.33334,-0.08,0.08,7.4945,outside_expected_envelope
27,Demo_Crew_01,T3_post,coactivation_edge_count,11.00000,-2.50,2.50,7.4194,outside_expected_envelope
57,Demo_Crew_02,T3_post,max_node_activation,0.42000,-0.10,0.10,7.0822,outside_expected_envelope
18,Demo_Crew_01,T2_inflight,active_domain_fraction,0.83333,-0.20,0.20,7.0259,outside_expected_envelope


## 10. Score hazard-context deltas against the envelope

Hazard-context delta envelope **does not measure exposure**. It calibrates whether the hazard-context mapping changed more than expected under reference variability.

In [6]:
if hazard_deltas is not None:
    hazard_scores = score_hazard_deltas_against_envelope(hazard_deltas, envelope)
    hazard_scores.to_csv(TABLES / 'reference_calibrated_hazard_delta_scores.csv', index=False)
    print('Position counts:', hazard_scores['envelope_position'].value_counts().to_dict())
    display(hazard_scores[hazard_scores['envelope_position'] == 'outside_expected_envelope']
            .sort_values('envelope_exceedance', ascending=False)[
        ['subject_id', 'timepoint', 'hazard', 'delta_hazard_relevance',
         'lower_bound', 'upper_bound', 'robust_z']].head(12))
else:
    hazard_scores = None
    print('Hazard delta table not available — hazard-context envelope scoring skipped.')


Position counts: {'within_expected_envelope': 28, 'outside_expected_envelope': 21, 'near_envelope_boundary': 1}


,subject_id,timepoint,hazard,delta_hazard_relevance,lower_bound,upper_bound,robust_z
16,Demo_Crew_01,T3_post,isolation_and_confinement,0.55000,-0.12,0.12,7.4194
41,Demo_Crew_02,T3_post,isolation_and_confinement,0.43000,-0.12,0.12,5.8006
17,Demo_Crew_01,T3_post,distance_from_earth,0.40000,-0.12,0.12,5.3959
11,Demo_Crew_01,T2_inflight,isolation_and_confinement,0.40000,-0.12,0.12,5.3959
15,Demo_Crew_01,T3_post,space_radiation,0.38539,-0.12,0.12,5.1988
18,Demo_Crew_01,T3_post,gravity_fields,0.37250,-0.12,0.12,5.0250
19,Demo_Crew_01,T3_post,hostile_closed_environments,0.37050,-0.12,0.12,4.9980
42,Demo_Crew_02,T3_post,distance_from_earth,0.35000,-0.12,0.12,4.7214
44,Demo_Crew_02,T3_post,hostile_closed_environments,0.34750,-0.12,0.12,4.6877
36,Demo_Crew_02,T2_inflight,isolation_and_confinement,0.33000,-0.12,0.12,4.4516


## 11. Build Phase 8 summary

One row per subject-timepoint, counting outside-envelope deltas per layer and assigning an overall envelope flag.

In [7]:
phase8_summary = build_phase8_envelope_summary(node_scores, graph_scores, hazard_scores)
phase8_summary.to_csv(TABLES / 'phase8_reference_calibrated_summary.csv', index=False)
phase8_summary


,subject_id,timepoint,mission_phase,n_outside_node_envelope,n_outside_graph_envelope,n_outside_hazard_envelope,top_outside_domain,top_outside_graph_metric,top_outside_hazard_context,overall_envelope_flag,interpretation
0,Demo_Crew_01,T0_baseline,baseline,0,0,0,n/a,n/a,n/a,within_expected_envelope,"At T0_baseline (baseline), all scored deltas r..."
1,Demo_Crew_01,T1_pre,pre_mission,0,0,0,n/a,n/a,n/a,within_expected_envelope,"At T1_pre (pre_mission), all scored deltas rem..."
2,Demo_Crew_01,T2_inflight,inflight,6,7,5,Body composition / physical status,n_active_domains,isolation_and_confinement,outside_expected_envelope,"At T2_inflight (inflight), 18 baseline-relativ..."
3,Demo_Crew_01,T3_post,postflight,6,7,5,Body composition / physical status,coactivation_edge_count,isolation_and_confinement,outside_expected_envelope,"At T3_post (postflight), 18 baseline-relative ..."
4,Demo_Crew_01,T4_recovery,recovery,0,2,1,n/a,total_node_activation,isolation_and_confinement,outside_expected_envelope,"At T4_recovery (recovery), 3 baseline-relative..."
5,Demo_Crew_02,T0_baseline,baseline,0,0,0,n/a,n/a,n/a,within_expected_envelope,"At T0_baseline (baseline), all scored deltas r..."
6,Demo_Crew_02,T1_pre,pre_mission,0,0,0,n/a,n/a,n/a,within_expected_envelope,"At T1_pre (pre_mission), all scored deltas rem..."
7,Demo_Crew_02,T2_inflight,inflight,6,6,5,Metabolic regulation,n_active_domains,isolation_and_confinement,outside_expected_envelope,"At T2_inflight (inflight), 17 baseline-relativ..."
8,Demo_Crew_02,T3_post,postflight,6,7,5,Metabolic regulation,n_active_domains,isolation_and_confinement,outside_expected_envelope,"At T3_post (postflight), 18 baseline-relative ..."
9,Demo_Crew_02,T4_recovery,recovery,0,0,0,n/a,n/a,n/a,near_envelope_boundary,"At T4_recovery (recovery), one or more deltas ..."


## 12. Visualize envelope results

Each figure carries a guardrail caption: envelope exceedance means a larger-than-expected baseline-relative delta — **not diagnosis, not risk, not exposure**.

In [8]:
figs = {}
figs['domain'] = plot_domain_delta_envelope(
    node_scores, FIGURES / 'phase8_domain_delta_envelope.png')
figs['graph'] = plot_graph_metric_envelope(
    graph_scores, FIGURES / 'phase8_graph_metric_envelope.png')
figs['hazard'] = plot_hazard_delta_envelope(
    hazard_scores, FIGURES / 'phase8_hazard_delta_envelope.png')
figs['heatmap'] = plot_envelope_exceedance_heatmap(
    phase8_summary, FIGURES / 'phase8_envelope_exceedance_heatmap.png')
figs['overview'] = plot_reference_envelope_overview(
    envelope, FIGURES / 'phase8_reference_envelope_overview.png')
for name, p in figs.items():
    print(f'{name}: {Path(p).name}')


domain: phase8_domain_delta_envelope.png
graph: phase8_graph_metric_envelope.png
hazard: phase8_hazard_delta_envelope.png
heatmap: phase8_envelope_exceedance_heatmap.png
overview: phase8_reference_envelope_overview.png


**Figure captions (reviewer-facing):** envelope exceedance = larger-than-expected baseline-relative delta. It is **not** diagnosis, **not** a health risk score, and **not** exposure measurement. The reference envelope overview shows which features are naturally more variable (wider envelopes).

## 13. Generate Phase 8 report

In [9]:
report = generate_phase8_report(
    phase8_summary, node_scores, graph_scores, hazard_scores,
    envelope_df=envelope, data_provenance_note=provenance_note,
)
report_path = REPORTS / 'phase8_reference_calibrated_trajectory_report.txt'
report_path.write_text(report, encoding='utf-8')
print(f'Report written to {report_path.name}')
print(report)


Report written to phase8_reference_calibrated_trajectory_report.txt
NeuroBridge-S4 Graph Learning
Phase 8 — Reference-Calibrated Trajectory Envelope

OVERVIEW
------------------------------------------------------------------------------
Phase 8 keeps within-subject self-baseline tracking as the primary method and adds a reference-calibrated variability envelope. It estimates whether each subject's own baseline-relative graph change is small, moderate, or unusually large relative to expected variability.

WHAT THE ENVELOPE IS
------------------------------------------------------------------------------
  The reference envelope does not define whether a person is healthy or unhealthy. It calibrates how large a within-subject graph change is relative to expected variability in available proxy or analog data.

WHAT THE ENVELOPE IS NOT
------------------------------------------------------------------------------
  - It is not a diagnosis or abnormality label.
  - It is not a health risk 

## 14. Conclusion

Phase 8 strengthens self-baseline trajectory tracking by adding a variability envelope. The method still prioritizes individual change over time, while using reference/analog data only for calibration.

Outside-envelope means a baseline-relative change is larger than expected under the current calibration data — a **candidate for expert review**, not diagnosis, risk scoring, treatment guidance, or exposure measurement.

**Next phase recommendation:** Phase 9 — Interactive longitudinal dashboard, letting reviewers explore trajectories, attribution, and envelope exceedance together.